In [1]:
import pandas as pd

In [2]:
#Read datasets
pao1_curated = pd.read_csv('dataFiles/PAO1_curated.tsv', sep='\t')
pao1_go = pd.read_csv('dataFiles/PAO1_gene_ontology.tsv', sep='\t')
pao1_cog = pd.read_csv('dataFiles/PAO1_cog.tsv', sep='\t')

#Get curated accession and category
pao1_go = pao1_go[['PAO1 ID','Name','Category']].drop_duplicates().dropna(subset=['Name'])
pao1_go = pao1_go.rename(columns={'PAO1 ID':'PAO1_ID','Name':'term','Category':'category'})
pao1_go['pathway'] = 'GO'

pao1_cog = pao1_cog[['PAO1 ID','Name','Category']].drop_duplicates().dropna(subset=['Name'])
pao1_cog = pao1_cog.rename(columns={'PAO1 ID':'PAO1_ID','Name':'term','Category':'category'})
pao1_cog['pathway'] = 'COG'

pao1_pseudocap = pao1_curated.copy()[['PAO1_ID','PseudoCap']].drop_duplicates().dropna(subset=['PseudoCap'])
pao1_pseudocap['PseudoCap'] = pao1_pseudocap['PseudoCap'].str.split(';')
pao1_pseudocap = pao1_pseudocap.explode('PseudoCap')
pao1_pseudocap['PseudoCap'] = pao1_pseudocap['PseudoCap'].str.strip()
pao1_pseudocap = pao1_pseudocap[pao1_pseudocap['PseudoCap'] != '']
pao1_pseudocap = pao1_pseudocap.rename(columns={'PseudoCap':'term'})
pao1_pseudocap = pao1_pseudocap.drop_duplicates().dropna(subset=['term'])
pao1_pseudocap['category'] = None
pao1_pseudocap['pathway'] = 'PseudoCap'

pao1_KEGG = pao1_curated.copy()[['PAO1_ID','KEGG']].drop_duplicates().dropna(subset=['KEGG'])
pao1_KEGG['KEGG'] = pao1_KEGG['KEGG'].str.split(';')
pao1_KEGG = pao1_KEGG.explode('KEGG')
pao1_KEGG['KEGG'] =pao1_KEGG['KEGG'].str.strip()
pao1_KEGG =pao1_KEGG[pao1_KEGG['KEGG'] != '']
pao1_KEGG = pao1_KEGG.rename(columns={'KEGG':'term'})
pao1_KEGG = pao1_KEGG.drop_duplicates().dropna(subset=['term'])
pao1_KEGG['category'] = None
pao1_KEGG['pathway'] = 'KEGG'
pao1_KEGG

#Merge datasets
pao1_go_cog = pd.concat([pao1_go, pao1_cog, pao1_pseudocap, pao1_KEGG]).drop_duplicates(ignore_index=True)
pao1_go_cog['category'] = pao1_go_cog['category'].str.replace({'_':' '})
pao1_go_cog.to_csv('dataFiles/pathways.tsv',sep='\t',index=False)